In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import pickle
import os
from sklearn.model_selection import train_test_split
import csv

Matplotlib is building the font cache; this may take a moment.


In [3]:
import torchvision

In [4]:
import torchvision.transforms as transforms

Setting device on gpu

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')

In [10]:
device

device(type='cpu')

In [8]:
transform = transforms.Compose([
    transforms.ToTensor(),
    # Optional: Normalize with the mean and standard deviation of CIFAR-10 dataset
    # transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# 2. Download and load the training and test datasets
train_dataset = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True,
    download=True, 
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data', 
    train=False,
    download=True, 
    transform=transform
)

100%|██████████| 170M/170M [36:54<00:00, 77.0kB/s]   


Extracting ./data/cifar-10-python.tar.gz to ./data


/opt/anaconda3/envs/ml_env/lib/python3.11/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Files already downloaded and verified


Loading Dataset

In [13]:
class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        
        # --- THE MAIN PATHWAY ---
        # First layer: changes dimensions if stride > 1
        self.my_conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.my_bn1 = nn.BatchNorm2d(out_channels)
        
        # Second layer: keeps dimensions exactly the same
        self.my_conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.my_bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        # --- THE SHORTCUT PATHWAY ---
        # Instead of an empty nn.Sequential(), we use a simple flag to remember if shapes match
        self.needs_shape_fix = (stride != 1) or (in_channels != out_channels)
        
        if self.needs_shape_fix:
            # A 1x1 convolution is used as a "shrink ray" to match the main path's size
            self.fix_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False)
            self.fix_bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        # Step 1: Make a copy of the input data (The Skip Connection)
        original_input = x
        
        # Step 2: Push the input through the Main Pathway
        # Layer 1: Conv -> BatchNorm -> ReLU
        out = self.my_conv1(x)
        out = self.my_bn1(out)
        out = self.relu(out)
        
        # Layer 2: Conv -> BatchNorm (No ReLU here yet!)
        out = self.my_conv2(out)
        out = self.my_bn2(out)
        
        # Step 3: Check if our original copy needs its size fixed to match 'out'
        if self.needs_shape_fix:
            original_input = self.fix_conv(original_input)
            original_input = self.fix_bn(original_input)
            
        # Step 4: The Core ResNet Magic — Add the original copy back to the changed output
        out = out + original_input
        
        # Step 5: Final activation function before sending it to the next layer
        out = self.relu(out)
        
        return out

In [ ]:
# class BeginnerRegularBlock(nn.Module):
#     def __init__(self, channels):
#         super().__init__()
#         # Main path layers (keeps size and channels completely identical)
#         self.conv1 = nn.Conv2d(3, channels, kernel_size=3, stride=1, padding=1, bias=False)
#         self.bn1 = nn.BatchNorm2d(channels)
        
#         self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1, bias=False)
#         self.bn2 = nn.BatchNorm2d(channels)
#         self.relu = nn.ReLU()

#     def forward(self, x):
#         # 1. Save a clean duplicate copy of the input
        
        
#         # 2. Go through the first layer
#         out = self.conv1(x)
#         out = self.bn1(out)
#         out = self.relu(out)
#         backup_copy = out
        
#         # 3. Go through the second layer
#         out = self.conv2(out)
#         out = self.bn2(out)
        
#         # 4. Math rules allow this directly because shapes match perfectly!
#         out = out + backup_copy
#         out = self.relu(out)
        
#         return out

In [14]:
class ResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        
        # --- STEP 1: PRE-RESNET LAYER (The Front Door) ---
        # Takes standard 3-channel RGB image and expands it to 32 channels
        self.front_conv = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1, bias=False)
        self.front_bn = nn.BatchNorm2d(32)
        self.relu = nn.ReLU()
        
        # --- STEP 2: STAGE 1 (Channels: 32, Image size remains 32x32) ---
        # No shape changes happen here, so stride=1 everywhere.
        self.stage1_block1 = Residual(in_channels=32, out_channels=32, stride=1)
        self.stage1_block2 = Residual(in_channels=32, out_channels=32, stride=1)
        
        # --- STEP 3: STAGE 2 (Channels: 32 -> 64, Image size shrinks to 16x16) ---
        # block1 triggers your 'needs_shape_fix' inside Residual because channels change and stride=2.
        self.stage2_block1 = Residual(in_channels=32, out_channels=64, stride=2)
        self.stage2_block2 = Residual(in_channels=64, out_channels=64, stride=1)
        
        # --- STEP 4: STAGE 3 (Channels: 64 -> 128, Image size shrinks to 8x8) ---
        # block1 triggers your 'needs_shape_fix' again because channels change and stride=2.
        self.stage3_block1 = Residual(in_channels=64, out_channels=128, stride=2)
        self.stage3_block2 = Residual(in_channels=128, out_channels=128, stride=1)
        
        # --- STEP 5: FINAL EXIT OUTLETS ---
        # Global Average Pooling squashes the 8x8 feature maps down into a 1x1 size
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        # Fully connected layer maps the 128 features to the final classification scores
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        # 1. Processing the raw input image at the front door
        out = self.front_conv(x)
        out = self.front_bn(out)
        out = self.relu(out)
        
        # 2. Flow through Stage 1
        out = self.stage1_block1(out)
        out = self.stage1_block2(out)
        
        # 3. Flow through Stage 2 (Doubles channels, cuts spatial size in half)
        out = self.stage2_block1(out)
        out = self.stage2_block2(out)
        
        # 4. Flow through Stage 3 (Doubles channels, cuts spatial size in half again)
        out = self.stage3_block1(out)
        out = self.stage3_block2(out)
        
        # 5. Flatten the matrix into a vector and guess the class label
        out = self.global_pool(out)
        out = torch.flatten(out, start_dim=1) # Changes shape from [Batch, 128, 1, 1] to [Batch, 128]
        out = self.classifier(out)
        
        return out

In [15]:
# Define batch size
BATCH_SIZE = 128

# Create DataLoader for training set
train_loader = torch.utils.data.DataLoader(
    dataset=train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=2
)

# Create DataLoader for testing set
test_loader = torch.utils.data.DataLoader(
    dataset=test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=2
)

print(f"Training batches: {len(train_loader)}")
print(f"Testing batches: {len(test_loader)}")

Training batches: 391
Testing batches: 79


In [16]:
# 1. Instantiate your ResNet model
model = ResNet(num_classes=10)
model = model.to(device)

# 2. Define Loss function (CrossEntropy is standard for multiclass classification)
criterion = nn.CrossEntropyLoss()

# 3. Define Optimizer (Adam optimizer with a standard starting learning rate)
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)

ResNet(
  (front_conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (front_bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (stage1_block1): Residual(
    (my_conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (my_bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (my_conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (my_bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (stage1_block2): Residual(
    (my_conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (my_bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (my_conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (my_bn2): BatchNorm2d(32, eps=1e-05, momentum

In [17]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train() # Set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in dataloader:
        # Move inputs and labels to the configured device (GPU/CPU)
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Zero out existing gradients from the last step
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass (Backpropagation)
        loss.backward()
        
        # Update weights
        optimizer.step()
        
        # Track statistics
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    return epoch_loss, epoch_acc

In [18]:
def evaluate_model(model, dataloader, criterion, device):
    model.eval() # Set model to evaluation mode
    running_loss = 0.0
    correct = 0
    total = 0
    
    # Disable gradient tracking to save memory and compute speed
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    return epoch_loss, epoch_acc

In [ ]:
# Configure number of epochs
EPOCHS = 10

# Lists to log metrics for final plotting
train_losses, train_accuracies = [], []
test_losses, test_accuracies = [], []

print("Starting training process...")
for epoch in range(EPOCHS):
    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    # Evaluate
    test_loss, test_acc = evaluate_model(model, test_loader, criterion, device)
    
    # Save statistics
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] -> "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% || "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")
          
print("Training Complete!")

Starting training process...
Epoch [1/10] -> Train Loss: 1.2668 | Train Acc: 53.63% || Test Loss: 1.1902 | Test Acc: 57.40%


In [ ]:
plt.figure(figsize=(12, 5))

# Plot Losses
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss', color='blue')
plt.plot(test_losses, label='Test Loss', color='orange')
plt.title('Loss History')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Plot Accuracies
plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label='Train Accuracy', color='blue')
plt.plot(test_accuracies, label='Test Accuracy', color='orange')
plt.title('Accuracy History')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
os.makedirs('./models', exist_ok=True)
model_save_path = './models/custom_resnet_cifar10.pth'

torch.save(model.state_dict(), model_save_path)
print(f"Model checkpoint parameters successfully saved to {model_save_path}")